Finally, we write the code file itself.  Most of the text below will appear literally in the file.  Statements in single curly braces will be substituted by the `format` statement at the end of the string; double curly braces will just appear as literal braces in the output.

In the following, the ODE system directly integrates the quaternion rotors describing the spin directions and orbital angular velocity vector.  This allows us to reduce the number of variables in the system and automatically satisfy certain constraints, which should lead to better numerical behavior.

Note that in this formulation, `Phi` is a completely unnecessary variable.  It doesn't need to be computed because it is implict in the definition of the frame.  It is calculated here simply because it is easy to do, and might be handy to have around.  Of course, it could be derived from knowledge of the frame alone.

In [ ]:
# Always run this first
# NOTE: Do not define new basic variables in this notebook;
#       define them in Variables_Q.ipynb.  Use this notebook
#       to define new expressions built from those variables.

from __future__ import division # This needs to be here, even though it's in Variables.py
import sys
sys.path.insert(0, '..') # Look for modules in directory above this one
execfile('../Utilities/ExecNotebook.ipy')
UseQuaternions = True
from sympy import N
from sympy import Rational as frac # Rename for similarity to latex
execnotebook('../PNTerms/Variables_Q.ipynb')
from Utilities import CodeOutput
from textwrap import indent

In [ ]:
execfile('GenerateEvolution.ipy')
# Calculate waveform modes
ellMin=0
ellMax=8
def Express(atom,CodeConstructor):
    A=str(atom)
    for i in CodeConstructor.Atoms:
        try:
            if i.constant:
                j=0
                while j!=-1 and j+len(str(i))<=len(A):
                    j=A.find(str(i),j)
                    if j!=-1:
                        if j+len(str(i)) ==len(A) or not (A[j+len(str(i))].isalnum() or A[j+len(str(i))]=='_'):
                            B=A[0:j]+'Cons.{0}'.format(i)+A[j+len(str(i)):]
                            A=B
                            j=j+5+len(str(i))
                        else:
                            j=j+1
            elif not i.constant:
                j=0
                while j!=-1 and j+len(str(i))<=len(A):
                    j=A.find(str(i),j)
                    if j!=-1:
                        if j+len(str(i)) ==len(A) or not (A[j+len(str(i))].isalnum() or A[j+len(str(i))]=='_'):
                            if j==0:
                                B=A[0:j]+'Vars.{0}'.format(i)+A[j+len(str(i)):]
                                A=B
                                j=j+5+len(str(i))
                            elif not (A[j-1].isalnum() or A[j-1]=='_'):
                                B=A[0:j]+'Vars.{0}'.format(i)+A[j+len(str(i)):]
                                A=B
                                j=j+5+len(str(i))
                            else:
                                j=j+1
                        else:
                            j=j+1
        except:
            pass
    return A

InputArgumentsEvolution=CodeConstructor.CppInputArguments(11)
execnotebook('../PNTerms/PsiMModes.ipynb')
WaveformModeTerms = [WaveformModes_NoSpin, WaveformModes_Spin_Symm, WaveformModes_Spin_Asymm]
for Term in WaveformModeTerms:
    PNVariables.update(Term)
# For some reason I have to both overwrite these variables and pop them...
PNVariables.AddVariable('nHat', constant=True, fundamental=True,
                 substitution_atoms=[], substitution='Quaternions::xHat', datatype='Quaternions.Quaternion');
PNVariables.AddVariable('lambdaHat', constant=True, fundamental=True,
                substitution_atoms=[], substitution='Quaternions::yHat', datatype='Quaternions.Quaternion');
PNVariables.AddVariable('ellHat', constant=True, fundamental=True,
                substitution_atoms=[], substitution='Quaternions::zHat', datatype='Quaternions.Quaternion');
PNVariables.pop(nHat);
PNVariables.pop(lambdaHat);
PNVariables.pop(ellHat);

PNVariables.AddDerivedVariable('chiVec1', substitution_atoms=[chiVec1,wHat], substitution='mul(mul(conjugate(normalized(y[5]*wHat + y[6]*xHat + y[7]*yHat + y[8]*zHat)),mul(mul(mul(S_chi1,R_S1),zHat),mul(conjugate(R_S1),conjugate(S_chi1)))),normalized(y[5]*wHat + y[6]*xHat + y[7]*yHat + y[8]*zHat))');
PNVariables.AddDerivedVariable('chiVec2', substitution_atoms=[chiVec2,wHat], substitution='mul(mul(conjugate(normalized(y[5]*wHat + y[6]*xHat + y[7]*yHat + y[8]*zHat)),mul(mul(mul(S_chi2,R_S2),zHat),mul(conjugate(R_S2),conjugate(S_chi2)))),normalized(y[5]*wHat + y[6]*xHat + y[7]*yHat + y[8]*zHat))');
PNVariables.AddDerivedVariable('chi1_n', substitution_atoms=[chiVec1,nHat], substitution='np.array([chiVec1[1]])');
PNVariables.AddDerivedVariable('chi1_lambda', substitution_atoms=[chiVec1,lambdaHat], substitution='np.array([chiVec1[2]])');
PNVariables.AddDerivedVariable('chi1_ell', substitution_atoms=[chiVec1,ellHat], substitution='np.array([chiVec1[3]])');
PNVariables.AddDerivedVariable('chi2_n', substitution_atoms=[chiVec2,nHat], substitution='np.array([chiVec2[1]])');
PNVariables.AddDerivedVariable('chi2_lambda', substitution_atoms=[chiVec2,lambdaHat], substitution='np.array([chiVec2[2]])');
PNVariables.AddDerivedVariable('chi2_ell', substitution_atoms=[chiVec2,ellHat], substitution='np.array([chiVec2[3]])');

with open('PNPsiMModes.py', 'w') as f :
    f.write("""# File produced automatically by PNCodeGen.ipynb
import numpy as np
from numpy import dot, log, sqrt, pi
from numpy import euler_gamma as EulerGamma
from numba import jit, njit, float64, complex128
from numba.experimental import jitclass
import quaternionic

qmul = njit(quaternionic.algebra.multiply)
qexp=njit(quaternionic.algebra.exp)
qconj=njit(quaternionic.algebra.conj)
I=1j

@njit(cache=True)
def mul(A,B):
    C=np.empty(4)
    qmul(A,B,C)
    return C
    
@njit(cache=True)
def exp(A):
    B=np.empty(4)
    qexp(A,B)
    return B
    
@njit(cache=True)
def conjugate(A):
    B=np.empty(4)
    qconj(A,B)
    return B
    
@njit(cache=True)
def normalized(A):
    return A/np.linalg.norm(A)
""")

    PNWaveformModeOrder=HighestOrder
    PNWaveformModeOrderString = str(N(PNWaveformModeOrder,2)).replace('.','p')
    ModeExpressions0 = PNCollection()
    LM = [[ell,m] for ell in range(ellMin,ellMax+1) for m in range(-ell,ell+1)]
    for ell in range(ellMin, ellMax+1):
        for m in range(0, ell+1):
            Symm = (SymmetricWaveformModes([ell,m],PNOrder=PNWaveformModeOrder)).subs(log(v), logv)
            Asymm = (AsymmetricWaveformModes([ell,m],PNOrder=PNWaveformModeOrder)).subs(log(v), logv)
            ModeExpressions0.AddDerivedVariable('rhOverM_{0}_{1}_Symm'.format(ell,m),
                                                Symm, datatype='double')
            ModeExpressions0.AddDerivedVariable('rhOverM_{0}_{1}_Asymm'.format(ell,m),
                                               Asymm, datatype='double')
    CodeConstructor0 = CodeOutput.CodeConstructor(PNVariables, ModeExpressions0)
    f.write("""
ConsSpec=[{ConsSpec}]
@jitclass(ConsSpec)
class Constants:
    def __init__(self,{ConsArgu}):
{ConsInit}

VarsSpec=[{VarsSpec}]
@jitclass(VarsSpec)
class Variables:
    def __init__(self,{VarsArgu}):
{VarsInit}

@njit(cache=True)
def Initialization(Cons, {InputArguments}): 
{Initializations}
""".format(ConsSpec=CodeConstructor0.ConsSpec(),
           ConsArgu=CodeConstructor0.ConsArgu(),
           ConsInit=CodeConstructor0.ConsInit(),
           VarsSpec=CodeConstructor0.VarsSpec(),
           VarsArgu=CodeConstructor0.VarsArgu(),
           VarsInit=CodeConstructor0.VarsInit(),
           InputArguments=InputArgumentsEvolution,
           Initializations=CodeConstructor0.CppInitializationModes(4)))
    lenConsReal=0
    lenVarsReal=0
    lenConsComplex=0
    lenVarsComplex=0
    for atom in CodeConstructor0.Atoms:
        if atom.constant and str(atom).find('hHat')==-1:
            lenConsReal+=1
        elif not atom.constant and str(atom).find('hHat')==-1:
            lenVarsReal+=1
        elif atom.constant and str(atom).find('hHat')!=-1:
            lenConsComplex+=1
        elif not atom.constant and str(atom).find('hHat')!=-1:
            lenVarsComplex+=1
    
    for PNWaveformModeOrder in [frac(n,2) for n in range(0,2*int(HighestOrder)+1)]:
        print("Working on {0} PN...".format(PNWaveformModeOrder))
        PNWaveformModeOrderString = str(N(PNWaveformModeOrder,2)).replace('.','p')
        ModeExpressions = PNCollection()                     
        LM = [[ell,m] for ell in range(ellMin,ellMax+1) for m in range(-ell,ell+1)]
        lenLM=len(LM)
        Evaluations = []
        for ell in range(ellMin, ellMax+1):
            for m in range(0, ell+1):
                Symm = (SymmetricWaveformModes([ell,m],PNOrder=PNWaveformModeOrder)).subs(log(v), logv)
                Asymm = (AsymmetricWaveformModes([ell,m],PNOrder=PNWaveformModeOrder)).subs(log(v), logv)
                ModeExpressions.AddDerivedVariable('rhOverM_{0}_{1}_Symm'.format(ell,m),
                                                   Symm, datatype='double')
                ModeExpressions.AddDerivedVariable('rhOverM_{0}_{1}_Asymm'.format(ell,m),
                                                   Asymm, datatype='double')
                code1 = "    Symm = {0}".format(Express(sympy.pycode(Symm),CodeConstructor0))
                code2 = "    Asymm = {0}".format(Express(sympy.pycode(Asymm),CodeConstructor0))
                code3 = "    modes[{2}] = (Symm + Asymm)[0]".format(ell, m, LM.index([ell,m]))
                code4 = "    modes[{2}] = ({3}np.conjugate(Symm - Asymm))[0]".format(ell, -m, LM.index([ell,-m]),
                                                                               ('' if ((ell%2)==0) else '-'))
                try:
                    if int(sympy.pycode(Symm))==0 and int(sympy.pycode(Asymm))==0:
                        pass
                except:
                    Evaluations.append("    # (ell, m) = ({0}, +/- {1})".format(ell, m))
                    Evaluations.append(code1)
                    Evaluations.append(code2)
                    Evaluations.append(code3)
                    if m!=0:
                        Evaluations.append(code4)
        CodeConstructor = CodeOutput.CodeConstructor(PNVariables, ModeExpressions)
        #[print(atom) for atom in CodeConstructor.Atoms if atom.constant]

        # Start the class, write the declarations, initializations, etc.
        f.write("""
@njit(cache=True)
def Recalculate_{PNWaveformModeOrderString}(Cons,Vars,y):
    Vars.v = np.array([y[0]])
    Vars.rfrak_chi1 = np.array([y[1],y[2]])
    Vars.rfrak_chi2 = np.array([y[3],y[4]])
{Evaluations}

@njit(cache=True)
def Modes_{PNWaveformModeOrderString}(modes,Cons,Vars):
{Computations}
""".format(PNWaveformModeOrderString=PNWaveformModeOrderString,
           Evaluations=CodeConstructor.CppEvaluationModes(4),
           Computations='\n'.join(Evaluations)))
        
    f.write("""
def Modes(wHat_i, xHat_i, yHat_i, zHat_i, M1_i, M2_i, v_i, S_chi1_i, S_chi2_i, frame, y, PNWaveformModeOrder=3.5) :
    Recalculate={{{ReCal}}}
    WaveformModes={{{ModeFunction}}}
    ModeData=np.zeros((len(y[0]),{lenLM}), dtype=complex)
    zz=np.array([0.0])
    z=np.array([0.0*1j])
    kappa = 1.0
    Cons=Constants({ConsInput})
    Vars=Variables({VarsInput})
    Initialization(Cons, wHat_i, xHat_i, yHat_i, zHat_i, M1_i, M2_i, v_i, S_chi1_i, S_chi2_i, kappa, kappa, kappa, kappa)
    for i in range(len(y[0])):
        Recalculate.get(2*PNWaveformModeOrder)(Cons,Vars,y[:,i])
        WaveformModes.get(2*PNWaveformModeOrder)(ModeData[i,:],Cons,Vars)
    return ModeData, [{ellMin},{ellMax}]
""".format(ReCal=',\n'.join(['        {0}:Recalculate_{1}'.format(i,str(N(i/2, 2)).replace('.','p')) for i in range(1+2*int(HighestOrder))]),
           ModeFunction=',\n'.join(['        {0}:Modes_{1}'.format(i,str(N(i/2, 2)).replace('.','p')) for i in range(1+2*int(HighestOrder))]),
           ConsInput=','.join(['zz' for i in range(lenConsReal)])+','+','.join(['z' for i in range(lenConsComplex)]),
           VarsInput=','.join(['zz' for i in range(lenVarsReal)])+','+','.join(['z' for i in range(lenVarsComplex)]),
           PNWaveformModeOrderString=PNWaveformModeOrderString,
           InputArgumentsEvolution=InputArgumentsEvolution,
           lenLM=lenLM,
           ellMin=ellMin,
           ellMax=ellMax))

print("All done")
from pathlib import Path
import runpy
runpy.run_path(str(Path('../../scripts/generate_emr22.py').resolve()), run_name='__main__')
